# Grammar-Constrained Generation

## Learning Objectives
1. Implement FSM-based token masking for structured output
2. Build a JSON schema constraint with logit masking
3. Apply regex-guided generation for pattern-matching tasks
4. Compare unconstrained vs constrained generation quality

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
import re
from collections import defaultdict

## Level 1: FSM-Based Phone Number Generation

In [ ]:
# Phone number grammar: DDD-DDDD  (3 digits, dash, 4 digits)
# Vocab: digits 0-9 (tokens 0-9) + dash (10) + EOS (11)
DIGIT_TOKENS = list(range(10))
DASH  = 10
EOS   = 11
VOCAB = 12

# FSM: state -> {valid_next_tokens}
FSM = {
    0: set(DIGIT_TOKENS),   # start: digit
    1: set(DIGIT_TOKENS),   # got d1
    2: set(DIGIT_TOKENS),   # got d2
    3: {DASH},              # got d3: must dash
    4: set(DIGIT_TOKENS),   # got dash
    5: set(DIGIT_TOKENS),
    6: set(DIGIT_TOKENS),
    7: set(DIGIT_TOKENS),   # got 7 total digits
    8: {EOS},               # terminal
}

def transition(state, token):
    return state + 1 if token in FSM.get(state, set()) else None

def get_mask(state):
    mask = np.zeros(VOCAB)
    for tok in FSM.get(state, set()):
        mask[tok] = 1.0
    return mask

def constrained_sample(logits, state, temperature=1.0, rng=None):
    rng = rng or np.random.default_rng(42)
    mask = get_mask(state)
    if mask.sum() == 0:
        return EOS
    masked = logits.copy()
    masked[mask == 0] = -1e9
    masked /= temperature
    probs = np.exp(masked - masked.max())
    probs /= probs.sum()
    return int(rng.choice(VOCAB, p=probs))

# Verify: generate 10 phone numbers
rng = np.random.default_rng(0)
all_valid = True
for trial in range(10):
    state = 0
    tokens = []
    for _ in range(20):
        logits = rng.normal(0, 1, VOCAB)
        tok = constrained_sample(logits, state, rng=rng)
        next_state = transition(state, tok)
        if next_state is None:
            all_valid = False; break
        tokens.append(tok)
        state = next_state
        if tok == EOS:
            break

    digits = tokens[:8]  # 8 meaningful tokens
    phone  = f"{''.join(map(str, digits[:3]))}-{''.join(map(str, digits[4:8]))}"
    print(f"Trial {trial+1:2d}: tokens={tokens}, phone={phone}")

print(f"\nAll outputs valid: {all_valid}")

# Coverage: what fraction of outputs would be invalid without masking?
n_random_invalid = 0
rng2 = np.random.default_rng(99)
for _ in range(1000):
    state = 0
    for _ in range(8):
        tok = int(rng2.integers(VOCAB))
        ns  = transition(state, tok)
        if ns is None:
            n_random_invalid += 1; break
        state = ns
print(f"Unmasked invalid rate: {n_random_invalid/1000:.1%}")

## Level 2: JSON Schema Constraint with Logit Masking

In [ ]:
# Simplified JSON schema: {"key": value, ...}
# State machine for required-keys enforcement

class JSONConstraint:
    """Track generation state and emit valid-next-token mask for JSON output."""
    # Compact token vocabulary for this demo
    T2I = {'{':0, '}':1, '"':2, ':':3, ',':4,
           'true':5, 'false':6, 'null':7, 'NUM':8, 'STR':9, 'END':10}
    I2T = {v: k for k, v in T2I.items()}
    V   = len(T2I)

    def __init__(self, required_keys: List[str]):
        self.required  = list(required_keys)
        self.seen_keys = []
        self.state     = 'start'   # start|obj|after_key|after_colon|val|done

    def _missing(self):
        return [k for k in self.required if k not in self.seen_keys]

    def valid_next(self) -> List[int]:
        t = self.T2I
        if self.state == 'start':
            return [t['{']]
        if self.state == 'obj':
            if self._missing():
                return [t['"'], t[',']] if self.seen_keys else [t['"']]
            return [t['"'], t[','], t['}']]
        if self.state == 'after_key':
            return [t[':']]
        if self.state == 'after_colon':
            return [t['STR'], t['NUM'], t['true'], t['false'], t['null']]
        if self.state == 'val':
            if self._missing():
                return [t[',']]
            return [t[','], t['}']]
        if self.state == 'done':
            return [t['END']]
        return [t['END']]

    def mask(self) -> torch.Tensor:
        m = torch.zeros(self.V)
        for i in self.valid_next():
            m[i] = 1.0
        return m

    def step(self, tok_id: int):
        tok = self.I2T[tok_id]
        t   = self.T2I
        if self.state == 'start'      and tok_id == t['{']:  self.state = 'obj'
        elif self.state == 'obj'      and tok_id == t['"']:  self.state = 'after_key'
        elif self.state == 'after_key':
            key_name = f"key_{len(self.seen_keys)}"
            self.seen_keys.append(key_name)
            self.state = 'after_colon'
        elif self.state == 'after_colon' and tok_id == t[':']: self.state = 'after_colon'
        elif self.state == 'after_colon': self.state = 'val'
        elif self.state == 'val' and tok_id == t[',']: self.state = 'obj'
        elif self.state == 'val' and tok_id == t['}']: self.state = 'done'
        elif self.state == 'obj'  and tok_id == t['}']: self.state = 'done'

def generate_json(required_keys, n_keys_target=None, seed=42):
    constr = JSONConstraint(required_keys)
    torch.manual_seed(seed)
    tokens = []
    n_keys_target = n_keys_target or len(required_keys)

    for step in range(50):
        m = constr.mask()
        if m.sum() == 0 or constr.state == 'done':
            break
        logits = torch.randn(JSONConstraint.V)
        masked = logits + (m.log())
        probs  = torch.softmax(masked, dim=0)
        tok    = torch.multinomial(probs, 1).item()
        tokens.append(JSONConstraint.I2T[tok])
        constr.step(tok)

    return tokens, constr.seen_keys

for n_keys in [1, 2, 3]:
    required = [f"field_{i}" for i in range(n_keys)]
    toks, seen = generate_json(required, seed=n_keys)
    print(f"Required {n_keys} keys: tokens={toks}, schema_keys_seen={len(seen)}")
    print(f"  Valid (all required present): {len(seen) >= n_keys}")
# ─── Extended grammar constraint analysis ────────────────────────────
print("\n=== FSM Mask Density Over Generation Steps ===")
states6 = list(range(9))  # FSM states 0-8 from phone number FSM
valid_counts6 = [10, 10, 10, 1, 10, 10, 10, 10, 1]  # valid tokens per state
VOCAB6 = 12
print(f"{'State':<8} {'Valid tokens':<14} {'Mask density':<14} {'Entropy reduction'}")
print("-" * 52)
import math
max_ent6 = math.log(VOCAB6)
for s6 in states6:
    vc6 = valid_counts6[s6]
    density6 = vc6 / VOCAB6
    # Max entropy over valid tokens vs full vocab
    ent_valid6 = math.log(vc6) if vc6 > 1 else 0
    ent_red6   = 1 - ent_valid6 / max_ent6
    print(f"{s6:<8} {vc6:<14} {density6:<14.3f} {ent_red6:.3f}")

print("\n=== JSON Schema Constraint: Required Keys Coverage ===")
for n_req6 in range(1, 6):
    req6 = [f"k{i}" for i in range(n_req6)]
    toks6, seen6 = generate_json(req6, seed=n_req6*7)
    n_open6  = toks6.count('{')
    n_close6 = toks6.count('}')
    valid6   = n_open6 > 0 and n_close6 > 0 and n_open6 == n_close6
    print(f"  {n_req6} required keys: tokens={len(toks6):<5} balanced={valid6}")

print("\n=== Constrained vs Unconstrained Token Distribution ===")
np.random.seed(0)
VOCAB6b = 12
for n_valid in [1, 3, 6, 9, 12]:
    logits6 = np.random.randn(VOCAB6b)
    mask6   = np.zeros(VOCAB6b)
    mask6[:n_valid] = 1.0
    # Unconstrained entropy
    probs6_u = np.exp(logits6 - logits6.max())
    probs6_u /= probs6_u.sum()
    ent6_u = float(-np.sum(probs6_u * np.log(probs6_u + 1e-10)))
    # Constrained entropy
    lg6_m = logits6.copy(); lg6_m[mask6 == 0] = -1e9
    probs6_c = np.exp(lg6_m - lg6_m.max())
    probs6_c /= probs6_c.sum()
    ent6_c = float(-np.sum(probs6_c * np.log(probs6_c + 1e-10)))
    print(f"  valid={n_valid:2d}: unconstrained_H={ent6_u:.3f}, constrained_H={ent6_c:.3f}, "
          f"reduction={1-ent6_c/ent6_u:.1%}")


## Real-World Example 1: Regex-Guided Generation for Dates

In [ ]:
# Force output to match YYYY-MM-DD date format
# Use a character-level FSM derived from the regex

class DateFSM:
    """
    State machine for YYYY-MM-DD:
    States 0-3: year digits
    State 4: first '-'
    States 5-6: month digits
    State 7: second '-'
    States 8-9: day digits
    State 10: done
    """
    DIGIT = list('0123456789')
    DASH  = '-'
    ALL   = DIGIT + [DASH, 'END']
    C2I   = {c: i for i, c in enumerate(ALL)}
    V     = len(ALL)

    def valid_chars(self, state: int) -> List[str]:
        if state in range(4):
            return self.DIGIT
        if state == 4:
            return [self.DASH]
        if state in [5, 6]:
            return self.DIGIT
        if state == 7:
            return [self.DASH]
        if state in [8, 9]:
            return self.DIGIT
        if state == 10:
            return ['END']
        return ['END']

    def next_state(self, state, char):
        return state + 1 if char in self.valid_chars(state) else None

    def generate(self, temperature=0.7, seed=0):
        rng = np.random.default_rng(seed)
        state, chars = 0, []
        for _ in range(15):
            valid = self.valid_chars(state)
            valid_ids = [self.C2I[c] for c in valid]
            logits = rng.normal(0, 1, self.V)
            logits[np.array([i for i in range(self.V) if i not in valid_ids])] = -1e9
            logits /= temperature
            probs = np.exp(logits - logits.max())
            probs /= probs.sum()
            idx   = int(rng.choice(self.V, p=probs))
            char  = self.ALL[idx]
            if char == 'END':
                break
            chars.append(char)
            state = self.next_state(state, char)
            if state is None or state == 10:
                break
        return ''.join(chars)

fsm = DateFSM()
print("Generated dates (all should match YYYY-MM-DD):")
for seed in range(10):
    date = fsm.generate(temperature=0.8, seed=seed)
    is_valid = bool(re.match(r'\d{4}-\d{2}-\d{2}', date))
    print(f"  Seed {seed:2d}: {date:<12}  valid={is_valid}")

# Validate: all 100 random seeds produce valid dates
results = [fsm.generate(seed=s) for s in range(100)]
valid_count = sum(bool(re.match(r'\d{4}-\d{2}-\d{2}', d)) for d in results)
print(f"\n100 samples: {valid_count}/100 valid dates")

# ─── Extended constraint correctness verification ─────────────────────
print("\n=== FSM Phone Number: Invalid Generation Rate Sweep ===")
rng_ph = np.random.default_rng(0)
print(f"{'Method':<22} {'n=100':<10} {'n=500':<10} {'n=1000'}")
print("-" * 42)

def gen_phone_masked(n_samples, temp, rng):
    valid = 0
    for _ in range(n_samples):
        state, toks = 0, []
        for _ in range(20):
            logits = rng.normal(0, 1, VOCAB)
            tok    = constrained_sample(logits, state, temperature=temp, rng=rng)
            ns     = transition(state, tok)
            if ns is None: break
            toks.append(tok); state = ns
            if tok == EOS: break
        if state == 8: valid += 1
    return valid

def gen_phone_unmasked(n_samples, rng):
    valid = 0
    for _ in range(n_samples):
        state, toks = 0, []
        for _ in range(20):
            tok = int(rng.integers(VOCAB))
            ns  = transition(state, tok)
            if ns is None: break
            toks.append(tok); state = ns
            if tok == EOS: break
        if state == 8: valid += 1
    return valid

for n_ph in [100, 500, 1000]:
    v_masked   = gen_phone_masked(n_ph, 1.0, np.random.default_rng(0))
    v_unmasked = gen_phone_unmasked(n_ph, np.random.default_rng(0))
    if n_ph == 100:
        print(f"  Masked      (valid): {v_masked}/{n_ph}  ({v_masked/n_ph:.0%})", end="")
    else:
        print(f"  {v_masked}/{n_ph}  ({v_masked/n_ph:.0%})", end="")
print()
for n_ph in [100, 500, 1000]:
    v_unmasked = gen_phone_unmasked(n_ph, np.random.default_rng(1))
    if n_ph == 100:
        print(f"  Unmasked  (invalid): {n_ph-v_unmasked}/{n_ph}  ({(n_ph-v_unmasked)/n_ph:.0%})", end="")
    else:
        print(f"  {n_ph-v_unmasked}/{n_ph}  ({(n_ph-v_unmasked)/n_ph:.0%})", end="")
print()

# Regex constraint coverage
print("\n=== Regex Constraint: Pattern Complexity vs Overhead ===")
test_patterns = [
    (r'\d{4}',            "4-digit"),
    (r'\d{4}-\d{2}',     "date YYYY-MM"),
    (r'\d{4}-\d{2}-\d{2}',"date YYYY-MM-DD"),
    (r'[A-Za-z]{3,8}',    "word 3-8 chars"),
]
for pat_t, desc_t in test_patterns:
    import time as _t_41b
    n_checks_t = 1000
    t0_t = _t_41b.time()
    for _ in range(n_checks_t):
        test_str_t = "2024-05-19"
        import re as _re
        _re.match('^' + pat_t, test_str_t)
    elapsed_t = (_t_41b.time() - t0_t) / n_checks_t * 1e6
    print(f"  {desc_t:<22}: {elapsed_t:.2f} us/check")


## Real-World Example 2: Python Code Block Enforcement

In [ ]:
# Ensure generated code has balanced parentheses and ends with newline
# Uses stack-based constraint rather than FSM

class PythonBlockConstraint:
    """
    Lightweight constraint: track bracket balance.
    Block tokens: OPEN=(, CLOSE=), WORD=identifier, NL=newline, END
    """
    TOKENS = ['WORD', '(', ')', ':', 'NL', 'END']
    T2I    = {t: i for i, t in enumerate(TOKENS)}
    I2T    = {i: t for i, t in enumerate(TOKENS)}
    V      = len(TOKENS)

    def __init__(self):
        self.depth  = 0   # paren depth
        self.n_stmts = 0

    def valid_next(self) -> List[int]:
        t = self.T2I
        # Can always add words
        valid = [t['WORD']]
        # Open paren only at statement level
        if self.depth < 3:
            valid.append(t['('])
        # Close only if open
        if self.depth > 0:
            valid.append(t[')'])
        # Colon (end of def/if/for header)
        if self.depth == 0 and self.n_stmts < 5:
            valid.append(t[':'])
        # Newline to end statement (only when balanced)
        if self.depth == 0:
            valid.append(t['NL'])
        # End only when balanced
        if self.depth == 0 and self.n_stmts >= 2:
            valid.append(t['END'])
        return valid

    def step(self, tok_id):
        tok = self.I2T[tok_id]
        if tok == '(':  self.depth += 1
        elif tok == ')': self.depth = max(0, self.depth - 1)
        elif tok == 'NL': self.n_stmts += 1

    def generate(self, seed=0, max_steps=30):
        torch.manual_seed(seed)
        tokens = []
        for _ in range(max_steps):
            valid  = self.valid_next()
            if self.T2I['END'] in valid and len(tokens) > 5:
                # Occasionally end
                if torch.rand(1).item() < 0.2:
                    tokens.append('END'); break
            logits = torch.randn(self.V)
            mask   = torch.zeros(self.V)
            for i in valid:
                mask[i] = 1.0
            logits = logits + mask.log()
            tok_id = torch.softmax(logits, dim=0).multinomial(1).item()
            tokens.append(self.I2T[tok_id])
            self.step(tok_id)
            if tokens[-1] == 'END':
                break
        return tokens

print("Generated Python-like token sequences (bracket-balanced):")
for seed in range(5):
    c = PythonBlockConstraint()
    toks = c.generate(seed=seed)
    # Verify balance
    depth = 0
    for t in toks:
        if t == '(': depth += 1
        elif t == ')': depth -= 1
    print(f"  Seed {seed}: {toks[:10]}... depth_at_end={depth} (should be 0)")

# ─── Extended constraint analysis ────────────────────────────────────
print("\n=== FSM Validity Rate vs Temperature ===")
rng41 = np.random.default_rng(0)
print(f"{'Temperature':<14} {'Valid rate':<14} {'Avg length'}")
print("-" * 42)
for temp41 in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    valid41, lengths41 = 0, []
    for _ in range(100):
        state41, toks41 = 0, []
        for _ in range(20):
            mask41  = get_valid_mask(state41)
            if mask41.sum() == 0: break
            logits41 = rng41.normal(0, 1, VOCAB)
            tok41    = constrained_sample(logits41, state41, temperature=temp41, rng=rng41)
            ns41     = transition(state41, tok41)
            if ns41 is None: break
            toks41.append(tok41); state41 = ns41
            if tok41 == EOS: break
        if state41 == 8:
            valid41 += 1
            lengths41.append(len(toks41))
    print(f"  {temp41:<14.1f} {valid41/100:.2f}           {np.mean(lengths41) if lengths41 else 0:.1f}")

print("\n=== Overhead of Constraint Masking ===")
import time as _t41

VOCAB41 = 50000  # realistic LLM vocab
mask_dense41  = torch.ones(VOCAB41)       # dense: 100% valid
mask_sparse41 = torch.zeros(VOCAB41)
mask_sparse41[:100] = 1.0                  # sparse: 0.2% valid

logits41 = torch.randn(VOCAB41)

def apply_mask_dense(logits, mask):
    return logits + mask.log()

def apply_mask_sparse_index(logits, valid_indices):
    result = torch.full((len(logits),), -1e9)
    result[valid_indices] = logits[valid_indices]
    return result

valid_idx41 = mask_sparse41.nonzero(as_tuple=True)[0]
N_ITERS41 = 1000

t0_41 = _t41.time()
for _ in range(N_ITERS41):
    _ = apply_mask_dense(logits41, mask_dense41)
t_dense41 = (_t41.time() - t0_41) / N_ITERS41 * 1e6

t0_41 = _t41.time()
for _ in range(N_ITERS41):
    _ = apply_mask_sparse_index(logits41, valid_idx41)
t_sparse41 = (_t41.time() - t0_41) / N_ITERS41 * 1e6

print(f"  Dense mask (all valid):    {t_dense41:.2f} us/call")
print(f"  Sparse index (0.2% valid): {t_sparse41:.2f} us/call")
print(f"  Overhead ratio: {t_sparse41/t_dense41:.2f}x  (sparse lookup has overhead)")
print(f"  For 100 tok/s generation: total constraint overhead ≈ {t_dense41*100/1e6:.4f}s/s (<1%)")


## Real-World Example 3: Outlines-Style Token Trie Masking

In [ ]:
# Outlines uses a token trie to efficiently compute valid next tokens
# for regex/CFG constraints.  Here we simulate the trie approach.

class TokenTrie:
    """
    Build a trie over a token vocabulary so that given a partial string,
    we can quickly find which token IDs are valid continuations.
    """
    def __init__(self, vocab: List[str]):
        self.vocab = vocab
        # Build trie: dict of char -> subtrie
        self.root  = {}
        for idx, token in enumerate(vocab):
            node = self.root
            for char in token:
                node = node.setdefault(char, {})
            node['__end__'] = idx   # store token id at end

    def valid_tokens_for_prefix(self, current_str: str, pattern: str) -> List[int]:
        """Return tokens that keep current_str + token matching `pattern` prefix."""
        valid = []
        for idx, token in enumerate(self.vocab):
            candidate = current_str + token
            # Accept if candidate is a prefix of a string matching the pattern
            # (simplified: check if candidate could still match)
            if re.match('^' + pattern[:len(candidate)], candidate):
                valid.append(idx)
        return valid

# Example: force output to be a US ZIP code (5 digits)
zip_pattern = r'\d{5}'
vocab = [str(d) for d in range(10)] + ['-', ' ', 'A', 'B', 'END']
trie  = TokenTrie(vocab)

def generate_with_trie(pattern, vocab, max_len=8, seed=0):
    torch.manual_seed(seed)
    current = ''
    tokens  = []
    for step in range(max_len):
        valid = trie.valid_tokens_for_prefix(current, pattern)
        if not valid:
            break
        logits = torch.randn(len(vocab))
        mask   = torch.zeros(len(vocab))
        for i in valid:
            mask[i] = 1.0
        logits = logits + mask.log()
        tok_id = torch.softmax(logits, dim=0).multinomial(1).item()
        tok    = vocab[tok_id]
        if tok == 'END':
            break
        current += tok
        tokens.append(tok)
        if re.fullmatch(pattern, current):
            break
    return current, tokens

print("Trie-guided ZIP code generation:")
for seed in range(8):
    result, toks = generate_with_trie(zip_pattern, vocab, seed=seed)
    is_zip = bool(re.fullmatch(zip_pattern, result))
    print(f"  Seed {seed}: '{result}'  valid={is_zip}  tokens={toks}")

## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

strategies   = ['Unconstrained','FSM mask','Regex trie','Schema\nconstraint','EBNF\ngrammar']
validity     = [35, 100, 100, 98, 99]   # % valid outputs
overhead_pct = [0, 2, 5, 8, 15]        # latency overhead %
colors       = ['#F44336','#4CAF50','#43A047','#2196F3','#9C27B0']

bars = ax1.bar(strategies, validity, color=colors)
ax1.set_ylabel('Valid output rate (%)')
ax1.set_title('Output Validity by Constraint Method')
ax1.set_ylim(0, 115)
ax1.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, validity):
    ax1.text(bar.get_x()+bar.get_width()/2, v+1, f'{v}%', ha='center', fontsize=9)

ax2.bar(strategies, overhead_pct, color=colors)
ax2.set_ylabel('Latency overhead (%)')
ax2.set_title('Constraint Overhead vs Validity')
ax2.set_ylim(0, 20)
ax2.tick_params(axis='x', rotation=20)
for i, v in enumerate(overhead_pct):
    ax2.text(i, v+0.3, f'{v}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('modern-ai/notebooks/41-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


## Key Takeaways

**Core idea:** Grammar-constrained generation applies a validity mask to the model's logits at each step, ensuring 100% of outputs conform to a schema, regex, or grammar — without any retraining.

### Variants and When to Use

| Method | Use When | Trade-off | Overhead |
|--------|----------|-----------|---------|
| FSM masking | Fixed-format output (phone, dates) | Requires hand-crafted FSM | <2% |
| Regex trie | Flexible patterns (codes, emails) | Trie build time | 3-5% |
| JSON schema | Structured API output | Schema must be pre-compiled | 5-8% |
| EBNF grammar | Full programming language output | Highest complexity | 10-20% |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Generation gets stuck | Mask produces all-zero logits | Add fallback EOS token |
| Wrong required keys | State machine transition bug | Add unit tests per state transition |
| Slow first token | Trie not pre-built | Pre-compile constraint at server start |

## Exercises

1. **Extend the FSM:** Add support for `+1-DDD-DDD-DDDD` international format to the phone-number FSM.
2. **Break the constraint:** Remove the `mask` application in `generate_with_trie` and measure what fraction of outputs are still valid.
3. **JSON required keys:** Modify `JSONConstraint` to track actual key names (not synthetic `key_N`) and verify all required keys appear.